# 09 — Invariant Procrustes baseline

Notebook chạy đúng baseline còn thiếu trong bảng gauge mechanism: một endpoint objective bất biến theo phép quay, giải Procrustes trực tiếp trên từng minibatch. Workflow giữ format của `00_main_results.ipynb`: cấu hình tập trung, một subprocess cho mỗi seed, teacher cache dùng chung, resume theo final-test record, và xuất mean ± sample standard deviation cùng một dòng LaTeX paper-ready.

Baseline tối ưu $\min_{R\in O(d_S)}\lVert Z_{\mathcal B}-Y_{\mathcal B}R\rVert_F^2$ trên từng batch. Implementation tương đương xoay student bằng nghiệm nghịch đảo, dùng `--endpoint_loss procrustes`; phép chọn $R$ được stop-gradient theo envelope theorem. Mọi yếu tố khác giữ giống full GATE-KD row trong configuration (c): fixed PCA subspace, native-space $H_0$ với $\lambda_{\mathrm{topo}}=0.5$, batch 128, 5 epochs và learning rate $7\times10^{-5}$.

`EXECUTE=False` là dry run. Kiểm tra ba commands ở cell 3 trước khi bật chạy.


In [2]:
# 1. Cấu hình thí nghiệm
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
TEACHER_MODEL = "Qwen/Qwen3-Embedding-0.6B"
STUDENT_MODEL = "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base"
TEACHER_POOLING = "last_token"
TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
SEEDS = [42, 43, 44]
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 7e-5
MAX_LENGTH = 256
NUM_WORKERS = 2
LAMBDA_H0 = 0.5
TOPO_BATCH_SIZE = 128
CUDA_VISIBLE_DEVICES = "0"
GPUS = None
MAX_PARALLEL_JOBS = 3
STOP_ON_ERROR = True
REQUIRE_ALL_SEEDS = True
EXECUTE = True
UPDATE_REPO = False
INSTALL_REQUIREMENTS = True
SAVE_TO_GOOGLE_DRIVE = False
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
RUN_NAME_OVERRIDE = None
RUN_NAME = RUN_NAME_OVERRIDE or f"invariant_procrustes_c_{len(SEEDS)}seeds_{RUN_STAMP}"
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert MAX_PARALLEL_JOBS >= 1
print(f"Run: {RUN_NAME}")
print(f"Seeds: {SEEDS}")


Run: invariant_procrustes_c_3seeds_20260906-215912
Seeds: [42, 43, 44]


In [3]:
# 2. Repo, dependencies, output, GPU và dữ liệu
import os
import subprocess
import sys
import torch

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file() and (cwd / "distiller.py").is_file():
    PROJECT_DIR = cwd
else:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if UPDATE_REPO:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"
RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu corpus: {TRAIN_DATA}"
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi chạy notebook.")
if hasattr(torch.cuda, "is_bf16_supported") and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU phải hỗ trợ BF16.")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
print(f"Corpus: {TRAIN_DATA}")
print(f"Output: {RUN_ROOT}")


cuda:0: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Corpus: /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv
Output: /content/embedding-kd/runs/invariant_procrustes_c_3seeds_20260906-215912


In [4]:
# 3. Tạo plan: một invariant-Procrustes job cho mỗi seed
import shlex

def build_command(seed):
    output_dir = RUN_ROOT / f"seed_{seed}"
    command = [
        sys.executable, str(PROJECT_DIR / "main.py"),
        "--method", "geoode",
        "--train_data", str(TRAIN_DATA),
        "--student_model", STUDENT_MODEL,
        "--teacher_model", TEACHER_MODEL,
        "--teacher_pooling", TEACHER_POOLING,
        "--student_pooling", "cls",
        "--batch_size", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--save_every", str(EPOCHS),
        "--lr", str(LEARNING_RATE),
        "--max_length", str(MAX_LENGTH),
        "--num_workers", str(NUM_WORKERS),
        "--seed", str(seed),
        "--eval_every", "0",
        "--pair_threshold_source", "test",
        "--save_dir", str(output_dir),
        "--cache_dir", str(CACHE_DIR),
        "--projection_type", "pca",
        "--pca_center_fit",
        "--no-pca_subtract_mean",
        "--endpoint_loss", "procrustes",
        "--no-gauge_align",
        "--gauge_refit_every", "0",
        "--lambda_end", "1",
        "--lambda_ctr", "0",
        "--lambda_gram", "0",
        "--lambda_topo", str(LAMBDA_H0),
        "--lambda_h1", "0",
        "--structural_loss", "h0",
        "--topo_teacher_source", "original",
        "--topo_batch_size", str(TOPO_BATCH_SIZE),
        "--no_eval_retrieval",
        "--no_wandb",
    ]
    return output_dir, command
JOBS = []
for seed in SEEDS:
    output_dir, command = build_command(seed)
    JOBS.append({"name": f"invariant_procrustes/seed_{seed}", "seed": seed, "output_dir": output_dir, "command": command, "log_path": output_dir / "train.log"})
for job in JOBS:
    command = job["command"]
    assert command[command.index("--endpoint_loss") + 1] == "procrustes"
    assert "--no-gauge_align" in command and command[command.index("--gauge_refit_every") + 1] == "0"
    assert command[command.index("--lambda_topo") + 1] == str(LAMBDA_H0)
print(f"Plan: {len(JOBS)} jobs")
for job in JOBS:
    print(f"[{job['name']}] {shlex.join(job['command'])}")


Plan: 3 jobs
[invariant_procrustes/seed_42] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --student_pooling cls --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --num_workers 2 --seed 42 --eval_every 0 --pair_threshold_source test --save_dir /content/embedding-kd/runs/invariant_procrustes_c_3seeds_20260906-215912/seed_42 --cache_dir /content/embedding-kd/runs/teacher_cache --projection_type pca --pca_center_fit --no-pca_subtract_mean --endpoint_loss procrustes --no-gauge_align --gauge_refit_every 0 --lambda_end 1 --lambda_ctr 0 --lambda_gram 0 --lambda_topo 0.5 --lambda_h1 0 --structural_loss h0 --topo_teacher_source original --topo_batch_size 128 --no_eval_retrieval --no_wandb
[invariant_procrustes/seed_43] /usr/bin/python3 /content/emb

In [5]:
# 4. Chạy/resume jobs và warm teacher cache trước khi fan-out
import json
sys.path.insert(0, str(PROJECT_DIR))
from src.job_runner import gpu_slots, run_jobs_parallel

def final_test_record(metrics_path):
    if not metrics_path.is_file(): return None
    found = None
    with metrics_path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                if record.get("test") and record.get("train") is None: found = record
    return found
pending, status = [], []
for job in JOBS:
    metrics = job["output_dir"] / "metrics.jsonl"
    if final_test_record(metrics) is not None:
        status.append({"seed": job["seed"], "status": "skipped_complete"})
    elif metrics.exists():
        raise RuntimeError(f"Run dở dang: {metrics}; dùng RUN_NAME mới hoặc archive riêng run này.")
    else:
        job["output_dir"].mkdir(parents=True, exist_ok=True)
        pending.append(job)
print(f"Complete: {len(status)}; pending: {len(pending)}")
if not EXECUTE:
    print("Dry run: đổi EXECUTE=True sau khi kiểm tra commands.")
elif pending:
    warm = list(pending[0]["command"])
    save_at = warm.index("--save_dir")
    del warm[save_at:save_at + 2]
    warm.extend(["--cache_only"])
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES, "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"}
    print(f"[CACHE] {shlex.join(warm)}")
    subprocess.run(warm, cwd=PROJECT_DIR, env=env, check=True)
    slots = gpu_slots(GPUS if GPUS is not None else [CUDA_VISIBLE_DEVICES], MAX_PARALLEL_JOBS)
    def on_finish(job, row):
        complete = row["returncode"] == 0 and final_test_record(job["output_dir"] / "metrics.jsonl") is not None
        payload = {"status": "complete" if complete else "failed", "wall_seconds": row["seconds"], "max_parallel": len(slots), "cuda_visible_devices": row["gpu"]}
        (job["output_dir"] / "runner_timing.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
        return {**row, **payload}
    rows = run_jobs_parallel(pending, cwd=PROJECT_DIR, env=env, slots=slots, stop_on_error=STOP_ON_ERROR, poll_seconds=30.0, on_finish=on_finish)
    status.extend(rows)
    failed = [row for row in rows if row.get("status") == "failed"]
    if failed: raise RuntimeError(f"{len(failed)} jobs failed; xem train.log.")


Complete: 0; pending: 3
[CACHE] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --teacher_pooling last_token --student_pooling cls --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --num_workers 2 --seed 42 --eval_every 0 --pair_threshold_source test --cache_dir /content/embedding-kd/runs/teacher_cache --projection_type pca --pca_center_fit --no-pca_subtract_mean --endpoint_loss procrustes --no-gauge_align --gauge_refit_every 0 --lambda_end 1 --lambda_ctr 0 --lambda_gram 0 --lambda_topo 0.5 --lambda_h1 0 --structural_loss h0 --topo_teacher_source original --topo_batch_size 128 --no_eval_retrieval --no_wandb --cache_only
[START 1/3] invariant_procrustes/seed_42 on GPU 0 (pid 6493) -> /content/embedding-kd/runs/invariant_procrustes_c_3seeds_20260906-215912/seed_42/train.

In [6]:
# 5. Collect mean ± sample std và in dòng LaTeX cho bảng
import numpy as np
import pandas as pd
from IPython.display import display
BENCHMARK_ORDER = ["banking77", "tweet", "emotion", "mrpc", "scitail", "wic", "sick", "sts12", "stsb"]
def benchmark_name(path):
    name = Path(path).stem
    return name[:-5] if name.endswith("_test") else name
def score_from_payload(family, raw):
    if family == "classification": return float(raw["f1"])
    if family == "pair": return float(raw["average_precision"])
    if family == "sts": return float(raw)
    raise KeyError(family)
rows, missing = [], []
for job in JOBS:
    record = final_test_record(job["output_dir"] / "metrics.jsonl")
    if record is None:
        missing.append(job["seed"]); continue
    test = record["test"]
    row = {"seed": job["seed"]}
    for family in ("classification", "pair", "sts"):
        for path, raw in test[family].items(): row[benchmark_name(path)] = score_from_payload(family, raw)
    row["avg_iod"] = float(test["summary"]["avg_iod"])
    row["avg_ood"] = float(test["summary"]["avg_ood"])
    row["avg_all"] = float(np.mean([row[name] for name in BENCHMARK_ORDER]))
    rows.append(row)
for seed in missing: print(f"[MISSING] seed={seed}")
if REQUIRE_ALL_SEEDS and missing and EXECUTE: raise RuntimeError(f"Thiếu completed seeds: {missing}")
by_seed = pd.DataFrame(rows)
if by_seed.empty:
    print("Chưa có completed runs để collect.")
else:
    metrics = [*BENCHMARK_ORDER, "avg_iod", "avg_ood", "avg_all"]
    mean = by_seed[metrics].mean()
    std = by_seed[metrics].std(ddof=1)
    count = by_seed[metrics].count()
    summary = pd.DataFrame({"mean": mean, "std": std, "n": count})
    by_seed.to_csv(RUN_ROOT / "invariant_procrustes_by_seed.csv", index=False)
    summary.to_csv(RUN_ROOT / "invariant_procrustes_mean_std.csv")
    display((100 * by_seed.set_index("seed")).style.format(precision=3))
    display((100 * summary[["mean", "std"]]).assign(n=summary["n"]).style.format({"mean": "{:.3f}", "std": "{:.3f}", "n": "{:.0f}"}))
    avg_mean, avg_std = 100 * mean["avg_all"], 100 * std["avg_all"]
    latex_row = f"Per-batch invariant Procrustes & Every step & \\gcell{{{avg_mean:.2f}}}{{{avg_std:.2f}}} \\\\"
    (RUN_ROOT / "gauge_mechanism_invariant_row.tex").write_text(latex_row + "\n", encoding="utf-8")
    print("Paper-ready row:")
    print(latex_row)
    print(f"Saved to: {RUN_ROOT}")


,banking77,emotion,tweet,mrpc,scitail,wic,sick,sts12,stsb,avg_iod,avg_ood,avg_all
seed,,,,,,,,,,,,
42,87.613,59.667,71.729,82.756,78.732,67.132,69.857,66.445,70.049,65.616,76.189,72.665
43,87.569,59.710,71.816,82.863,78.456,66.913,70.053,66.885,70.437,65.687,76.274,72.745
44,87.771,61.424,71.762,82.685,78.656,66.731,69.755,66.420,70.006,66.054,76.175,72.801


,mean,std,n
banking77,87.651,0.106,3
tweet,71.769,0.044,3
emotion,60.267,1.002,3
mrpc,82.768,0.089,3
scitail,78.615,0.143,3
wic,66.926,0.201,3
sick,69.888,0.152,3
sts12,66.584,0.261,3
stsb,70.164,0.237,3
avg_iod,65.786,0.235,3


Paper-ready row:
Per-batch invariant Procrustes & Every step & \gcell{72.74}{0.07} \\
Saved to: /content/embedding-kd/runs/invariant_procrustes_c_3seeds_20260906-215912
